# 02 - Full pipeline: every module against the synthetic Cu-Al fixture

Walk through the 16 published modules in dependency order, producing the same set of
AnalysisResult objects the `tests/test_synthetic_regression.py` E2E gate exercises. End
with a master inventory CSV + the four publication-figure scaffolds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from midas_defect.asterism import per_grain_asterism_tensor, edge_fraction_per_grain
from midas_defect.debye_waller import per_grain_B_factor
from midas_defect.distributions import friedel_pair_asymmetry, mackenzie_pdf
from midas_defect.energy import elastic_energy_density_cubic, volume_weighted_energy_per_variant
from midas_defect.gnd import per_grain_nye_tensor, scalar_gnd_from_inter_grain_misorientation
from midas_defect.line_profile import collect_per_grain_reflections, modified_wh_per_grain
from midas_defect.phases import FCC_SLIP_111_110, FCC_TWIN_111_112
from midas_defect.reports import write_master_inventory_csv, matrix_twin_summary_figure
from midas_defect.schmid import schmid_factor_per_grain, stratify_pairs_by_schmid_max
from midas_defect.spatial import epsilon_autocorrelation
from midas_defect.strain import twin_shear_projection_per_pair, von_mises_strain
from midas_defect.stress import per_grain_stress_cubic, von_mises
from midas_defect.thermodynamics import variant_specific_k2, taylor_implied_total_rho
from midas_defect.types import AnalysisResult, BootUnit, CrystalPhase
from midas_defect.variants import assign_variants_kmeans, find_sigma3_partners

from tests.conftest import synthetic_cu_al_dataset  # type: ignore
data = synthetic_cu_al_dataset.__wrapped__()
C11, C12, C44 = 169.0, 122.0, 75.3   # Cu single-crystal (GPa)

## 1. Variants + matched pairs

In [ ]:
var_out = assign_variants_kmeans(data['OM'], n_variants=2, n_init=10, phase=CrystalPhase.FCC)
labels = var_out['labels']
pairs = find_sigma3_partners(data['OM'], data['pos'], labels, k_NN=5, phase=CrystalPhase.FCC)

## 2. Schmid + tercile stratification

In [ ]:
schmid = schmid_factor_per_grain(data['OM'], data['loading_axis'], FCC_SLIP_111_110)
if pairs['pairs'].shape[0] >= 6:
    strat = stratify_pairs_by_schmid_max(pairs['pairs'], schmid)
    print('Schmid tier edges:', strat['tier_edges'])

## 3. Stress + strain + energy

In [ ]:
sigma = per_grain_stress_cubic(data['OM'], data['eps_sample'], C11, C12, C44)
sigma_vM = von_mises(sigma)
eps_eq = von_mises_strain(data['eps_sample'])
U = elastic_energy_density_cubic(data['OM'], data['eps_sample'], C11, C12, C44)
vw = volume_weighted_energy_per_variant(U, data['radii'], labels)
print(f'U(matrix) = {vw["U_mean_per_variant"][0]:.3e} Pa   U(twin) = {vw["U_mean_per_variant"][1]:.3e} Pa')
if pairs['pairs'].shape[0] > 0:
    proj = twin_shear_projection_per_pair(
        data['eps_sample'], data['OM'], pairs['pairs'], FCC_TWIN_111_112, data['loading_axis']
    )
    print('median dEps on active twin shear:', np.median(proj['dEps_twin_shear']))

## 4. Line-profile dislocation density

In [ ]:
entries = collect_per_grain_reflections(
    data['qs'], data['vals'], data['grain_of_voxel'], data['OM'], data['G_arr'],
    query_radius=0.20, min_voxels_per_refl=8,
)
wh = modified_wh_per_grain(entries, data['hkls'], burgers_length=data['burgers'])
rho = wh['rho_per_grain']
B = per_grain_B_factor(entries, data['hkls'], structure_factor_squared=lambda hkl: 1.0)
print(f'rho median:  {np.nanmedian(rho):.3e} m^-2')
print(f'B  median:   {np.nanmedian(B["B_per_grain"]):.2f} A^2')

## 5. Asterism + GND + Mackenzie + spatial

In [ ]:
Pn = np.zeros_like(data['qs'])  # mock nearest-Bragg for speed
M = per_grain_asterism_tensor(data['qs'], data['vals'], data['grain_of_voxel'], Pn,
                              np.ones_like(data['vals'], dtype=bool), n_grains=data['OM'].shape[0])
rho_GND = scalar_gnd_from_inter_grain_misorientation(
    data['OM'], data['pos'], burgers_length=data['burgers']
)
ac = epsilon_autocorrelation(eps_eq, data['pos'])
print('Mackenzie peak deg (cubic FZ):', np.linspace(0, 64, 200)[int(np.argmax(mackenzie_pdf(np.linspace(0, 64, 200), CrystalPhase.FCC)))])

## 6. Thermodynamics + master inventory

In [ ]:
rho_sat = {
    'matrix': float(np.nanpercentile(rho[labels == 0], 84)),
    'twin':   float(np.nanpercentile(rho[labels == 1], 84)),
}
mk = variant_specific_k2(rho_sat)
rho_taylor = taylor_implied_total_rho(7.0e8)
print(f'k2 ratio matrix/twin = {mk["k2_ratio_pairs"][("matrix", "twin")]:.2f}')
print(f'rho Taylor at 700 MPa = {rho_taylor:.2e}')

In [ ]:
# 7. Build a small AnalysisResult set and write a master inventory CSV.
def to_result(name, vals, units):
    rng = np.random.default_rng(abs(hash(name)) & 0xFFFFFFFF)
    boot = np.array([np.median(rng.choice(vals[np.isfinite(vals)], size=vals.size, replace=True))
                     for _ in range(128)])
    return AnalysisResult(
        name=name, units=units, boot_unit=BootUnit.GRAIN, n_boot=128,
        population_median=float(np.median(boot)),
        population_ci=(float(np.percentile(boot, 16)), float(np.percentile(boot, 84))),
        bootstrap_samples=boot, per_grain=vals,
    )

results = [
    to_result('rho_matrix', rho[labels == 0], 'm^-2'),
    to_result('rho_twin',   rho[labels == 1], 'm^-2'),
    to_result('U_matrix',   U[labels == 0],   'Pa'),
    to_result('U_twin',     U[labels == 1],   'Pa'),
]
write_master_inventory_csv(results, '/tmp/midas_defect_inventory.csv')
matrix_twin_summary_figure(results, '/tmp/midas_defect_summary.png', metrics=['rho', 'U'])
print('wrote /tmp/midas_defect_inventory.csv and /tmp/midas_defect_summary.png')